# Ordered Logistic Regression Results: Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata as an object (not via subscripting)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.
We'll enumerate the available record sets, fields, and columns using their `@id` to understand what data is available for loading.


In [ ]:
# List all record sets with their @id and available fields

print("Available record sets and fields:")
for record_set in dataset.metadata.record_sets:
    print(f"RecordSet @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '--')}")
    print(f"  Fields:")
    if 'fields' in record_set:
        for field in record_set['fields']:
            field_id = field.get('@id', None)
            label = field.get('name', '--')
            dtype = field.get('dataType', '--')
            print(f"    Field @id: {field_id}, Name: {label}, Type: {dtype}")
    else:
        print("    No fields information found.")
    print()

# For each recordset, we can preview some records (if available)
print("Record preview (first 1 record from each record set):\n")
for record_set in dataset.metadata.record_sets:
    rs_id = record_set['@id']
    print(f"RecordSet @id: {rs_id}")
    try:
        preview = next(dataset.records(record_set=rs_id))
        print(preview)
    except StopIteration:
        print("  No records found.")
    except Exception as e:
        print(f"  Could not load records for this record set: {e}")
    print("-")

## 3. Data Extraction
Load data from specific record set(s) into a DataFrame for analysis.
We'll demonstrate for **all available record sets**, referencing them by their `@id`.

In [ ]:
# Extract data from all detected record sets by @id

record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records_iter = dataset.records(record_set=rs_id)
    records = list(records_iter)
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for RecordSet {rs_id}")
    else:
        print(f"No records found for RecordSet {rs_id}")

if dataframes:
    # Pick the first available record set with data for further use
    main_record_set_id = list(dataframes.keys())[0]
    df_main = dataframes[main_record_set_id]
    print(f"\nColumns for RecordSet {main_record_set_id}:")
    print(df_main.columns.tolist())
    print(df_main.head())
else:
    print("No dataframes loaded. Check record sets or schema.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll:
  - Filter rows where a numeric field exceeds a threshold.
  - Normalize this numeric field.
  - Optionally, group by a categorical field if available.


In [ ]:
import numpy as np

# Identify a numeric field by @id
numeric_field_id = None
group_field_id = None
# Try to guess the first numeric field from current DataFrame

if not dataframes:
    print("No data available for EDA.")
else:
    df = df_main.copy()
    # Simple heuristic: treat float/int columns as candidates
    for col in df.columns:
        # Try to infer numeric columns
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            continue
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        numeric_field_id = num_cols[0]
        print(f"Selected numeric field (by @id): {numeric_field_id}")
    else:
        print("No numeric fields found for filtering/normalizing.")

    # Select a group field (categorical) -- pick first object type with limited unique values
    potential_groups = [col for col in df.columns if df[col].dtype == 'object' and df[col].nunique() > 1 and df[col].nunique() < 20]
    if potential_groups:
        group_field_id = potential_groups[0]
        print(f"Selected group field (by @id): {group_field_id}")

    # Proceed if we have at least a numeric field
    if numeric_field_id:
        # Drop missing values
        df_num = df.dropna(subset=[numeric_field_id]).copy()
        try:
            df_num[numeric_field_id] = df_num[numeric_field_id].astype(float)
        except Exception:
            pass

        threshold = df_num[numeric_field_id].mean()  # example: threshold set to mean
        filtered_df = df_num[df_num[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        mean_val = df_num[numeric_field_id].mean()
        std_val = df_num[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a field if available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric fields available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'df_num' in locals() and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_num[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df_num)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- In this notebook, we loaded and explored the Ordered Logistic Regression Results dataset using the `mlcroissant` library.
- We identified available record sets and fields by their `@id`, and demonstrated data loading, filtering, normalization, and visualization.
- Further, we applied simple analysis and visualized the distribution of a numeric field, as well as its aggregated behavior with regard to a key group attribute.

For more in-depth analysis, consult the dataset record set and Croissant documentation for schema details and rich metadata.